# Verifier agent
Checks a simplified text against its source. Calibrated against MedAESQA human labels.

In [1]:
!pip install -q google-genai google-adk pydantic scikit-learn pandas
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

## Verifier

In [9]:
import json, re, time
import numpy as np
import pandas as pd
from typing import Literal
from pydantic import BaseModel
from google import genai
from google.genai import types
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, cohen_kappa_score

MODEL = "gemini-3.1-flash-lite"
MIN_INTERVAL = 4.0                # seconds between calls (free tier is about 15/min);
client = genai.Client()

class ClaimCheck(BaseModel):
    claim: str
    evidence_quote: str
    verdict: Literal["supported", "unsupported", "contradicted"]

class VerifierReport(BaseModel):
    claim_checks: list[ClaimCheck]
    passed: bool
    reading_level_match: bool
    flagged_passages: list[str]

PROMPT = """You are a clinical faithfulness verifier. Judge the TEXT against the SOURCE only.

The SOURCE is a set of short excerpts from the papers the TEXT cites. The papers may
contain more than the excerpts show, so extra background detail is not an error.

SOURCE:
{source}

TEXT TO AUDIT:
{text}

1. Identify the main claim of the TEXT: the central point it makes. Ignore framing words
   like "Additionally" or "It is important to note".
2. Label each claim:
   supported = the source states it, entails it, or says the same thing in different words.
   contradicted = the source says something different or opposite. Check direction
                  (increase vs decrease), numbers, and whether an effect persists or stops.
                  Also contradicted: the TEXT states as definite what the source says is
                  uncertain, inconsistent or "may" happen; claims an effect that the source
                  says was not significant or happened "only" for something else; or says
                  the opposite of a negative finding ("does not", "no effect").
   unsupported = no quote found, including general background or definitions the source
                 never states.
3. passed = true only if BOTH hold:
   a) the main claim is supported by the source (the source must actually address the
      same topic and point, not merely mention related words), and
   b) no claim is contradicted.
   Peripheral detail that is merely unsupported does not make passed false.
4. reading_level_match = true if the wording fits roughly an 8th-grade reading level.
5. flagged_passages = exact text of any contradicted claim, plus the main claim if unsupported."""

In [12]:
_last = [0.0]
_cache = {}   # keyed on the prompt too, so editing PROMPT refreshes results automatically

def judge(source, text):
    key = (PROMPT, source, text)
    if key in _cache:
        return _cache[key]
    for attempt in range(4):
        wait = MIN_INTERVAL - (time.time() - _last[0])
        if wait > 0:
            time.sleep(wait)
        _last[0] = time.time()
        try:
            r = client.models.generate_content(
                model=MODEL,
                contents=PROMPT.format(source=source, text=text),
                config=types.GenerateContentConfig(
                    temperature=0, response_mime_type="application/json",
                    response_schema=VerifierReport),
            )
            _cache[key] = VerifierReport.model_validate_json(r.text)
            return _cache[key]
        except Exception as e:
            if "PerDay" in str(e):
                raise RuntimeError("Daily quota hit. Switch model or wait.") from e
            print(f"attempt {attempt + 1} failed: {e}")
            time.sleep(2 ** attempt * 3)
    raise RuntimeError("judge failed after retries")


def to_pipeline_json(report):
    return {
        "faithful": report.passed,
        "unsupported_claims": [c.claim for c in report.claim_checks if c.verdict != "supported"],
        "omissions": [],   # not checked in this version
        "reading_level_ok": report.reading_level_match,
    }


def calibrate(df):
    """df needs: source_text, answer_text, human_label (0/1)"""
    preds = [int(judge(r.source_text, r.answer_text).passed) for r in df.itertuples()]
    y = df["human_label"].astype(int).values
    p = np.array(preds)
    print(f"accuracy {accuracy_score(y, p):.3f} | kappa {cohen_kappa_score(y, p):.3f} | "
          f"false pos {int(((p == 1) & (y == 0)).sum())} | false neg {int(((p == 0) & (y == 1)).sum())}")
    return df.assign(verifier_label=preds)

## MedAESQA data


In [4]:
from google.colab import files

uploaded = files.upload()
fname = list(uploaded.keys())[0]
with open(fname) as f:
    data = json.load(f)

rows = []
for q in data:
    for mid, ans in q["machine_generated_answers"].items():
        for s in ans["answer_sentences"]:
            ca = s["citation_assessment"]
            if not ca:
                continue
            ev = [c["evidence_support"] for c in ca if c["evidence_support"]]
            if not ev:
                continue
            rels = {c["evidence_relation"] for c in ca}
            rows.append({
                "question_id": q["question_id"],
                "source_text": "\n".join(ev),
                "answer_text": re.sub(r"\s*\[[\d,\s]+\]", "", s["answer_sentence"]).strip(),
                "human_label": int("supporting" in rels and "contradicting" not in rels),
            })

df = (pd.DataFrame(rows)
        .drop_duplicates(subset=["question_id", "source_text", "answer_text"])
        .reset_index(drop=True))

# split by question so no question appears in both halves
train_q, test_q = train_test_split(df["question_id"].unique(), test_size=0.5, random_state=42)
train_df = df[df["question_id"].isin(train_q)].reset_index(drop=True)
test_df = df[df["question_id"].isin(test_q)].reset_index(drop=True)

print(len(df), "rows |", len(train_df), "train,", len(test_df), "test")
print("train labels:", train_df.human_label.value_counts().to_dict())

Saving medaesqa_v1.json to medaesqa_v1.json
3526 rows | 1857 train, 1669 test
train labels: {1: 1693, 0: 164}


## Calibration


In [13]:
N = 10

pos = train_df[train_df.human_label == 1].sample(N, random_state=0)
neg = train_df[train_df.human_label == 0].sample(N, random_state=0)
results = calibrate(pd.concat([pos, neg]))
results.to_csv("train_results.csv", index=False)

attempt 1 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


KeyboardInterrupt: 

In [ ]:
pd.set_option("display.max_colwidth", 250)

def show(rows):
    for r in rows.itertuples():
        rep = judge(r.source_text, r.answer_text)
        print("SOURCE:", r.source_text[:400])
        print("TEXT:  ", r.answer_text)
        print("HUMAN:", r.human_label, "| VERIFIER:", r.verifier_label)
        for c in rep.claim_checks:
            print(f"  [{c.verdict}] {c.claim}  <- \"{c.evidence_quote[:100]}\"")
        print("-" * 60)

wrong = results[results.human_label != results.verifier_label]
print(len(wrong), "disagreements")
show(wrong.head(6))

## Final check


In [ ]:
# pos = test_df[test_df.human_label == 1].sample(50, random_state=0)
# neg = test_df[test_df.human_label == 0].sample(50, random_state=0)
# final = calibrate(pd.concat([pos, neg]))
# final.to_csv("test_results.csv", index=False)

## ADK agent


In [ ]:
def build_verifier(model=MODEL):
    from google.adk.agents import LlmAgent, BaseAgent, SequentialAgent
    from google.adk.events import Event, EventActions

    instruction = PROMPT.replace("{source}", "{atoms}").replace("{text}", "{draft}")

    judge_llm = LlmAgent(
        name="VerifierLLM",
        model=model,
        instruction=instruction,
        output_schema=VerifierReport,
        output_key="verifier_raw",
        generate_content_config=types.GenerateContentConfig(temperature=0.0),
    )

    class VerdictBuilder(BaseAgent):
        async def _run_async_impl(self, ctx):
            report = VerifierReport.model_validate(ctx.session.state["verifier_raw"])
            yield Event(
                invocation_id=ctx.invocation_id,
                author=self.name,
                actions=EventActions(state_delta={"verdict": json.dumps(to_pipeline_json(report))}),
            )

    return SequentialAgent(name="Verifier",
                           sub_agents=[judge_llm, VerdictBuilder(name="VerdictBuilder")])